In [3]:
import sys, os

In [4]:
from common_utils.utils import chunks
from tqdm import tqdm
from matplotlib import pyplot as plt
from pylab import rcParams
import seaborn as sns

from IPython.display import display, HTML
import pandas as pd

import logging
from datetime import datetime, timedelta
import pandas as pd

from common_utils.notebook_utils import sign_notebook

pd.set_option('display.max_rows', 1000)
pd.set_option('display.max_columns', 200)
from pandas.io.json import json_normalize

rcParams['figure.figsize'] = 15, 8

sns.set(rc={'figure.figsize':(12,6)})

sns.set(style="whitegrid", color_codes=True)
sns.despine()


from dbutils.query_utils import get_interval_clauses, run_select
from dbutils.utils import get_nabu_payments_client, get_nabu_ledger_client
from dbutils.query_utils import get_select_query, run_select, get_interval_clauses
from customers_analytics.fraud_analysis.fraud_db_utils import get_fraud_client
from customers_analytics.nabu.nabu_db_utils import (
    get_nabu_client_internal, 
    get_nabu_backoffice_client, 
    get_nabu_client, 
    get_nabu_client_private, 
    get_nabu_backoffice_client_prod, 
    categorize_blockchain_error_reasons
)

INFO:root:Role:data-science Environment:internal


<Figure size 864x432 with 0 Axes>

In [5]:
client_internal = get_nabu_client_internal()

In [45]:
query = \
'''
with created_cards as (
SELECT 
     distinct
     id
    ,user_id
    ,date_trunc('day', created_at) as created_at_day
    ,to_timestamp(to_char(created_at, 'yyyy-MM-dd HH24'), 'yyyy-MM-dd HH24') as created_at_dayhour
    --,to_char(date_trunc('day', created_at), 'yyyy-MM-dd') created_at
FROM card_additions
WHERE lower(state) = 'created'
),
gr_created_cards as (
SELECT 
     user_id
    --,created_at
    ,created_at_day
    ,created_at_dayhour
    ,count(id) OVER cum_per_dayhour AS cum_amt_cards_per_dayhour
    ,count(id) OVER cum_last_seven_h AS cum_amt_cards_last_seven_h
FROM created_cards
WINDOW cum_last_seven_h AS (PARTITION BY user_id
                               ORDER BY created_at_dayhour ASC
                               RANGE BETWEEN INTERVAL '7 hours' PRECEDING
                                         AND CURRENT ROW),
       cum_per_dayhour AS (PARTITION BY user_id
                               ORDER BY created_at_dayhour ASC
                               ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)                              
--ORDER BY user_id, created_at, cum_amt asc
),
card_pay_fraud_stats as (
SELECT 
     user_id
    ,card_type
    ,usd_amount
    ,to_timestamp(to_char(created_at, 'yyyy-MM-dd HH24'), 'yyyy-MM-dd HH24') as created_at_dayhour
FROM card_payments_fraud_stats
),
gr_card_pay_fraud_stats as (
SELECT
     user_id
    ,card_type
    ,created_at_dayhour
    ,sum(usd_amount) as sum_usd_amount 
FROM card_pay_fraud_stats
GROUP BY user_id, card_type, created_at_dayhour
--order by created_at, sum_usd_amount desc
)
SELECT 
     gr_cards.user_id
    ,gr_cards.created_at_day 
    ,gr_cards.created_at_dayhour
    ,gr_cards.cum_amt_cards_per_dayhour
    ,gr_cards.cum_amt_cards_last_seven_h
    ,gr_stats.sum_usd_amount    
    ,gr_stats.card_type
FROM gr_created_cards as gr_cards
left join gr_card_pay_fraud_stats as gr_stats 
on gr_cards.user_id = gr_stats.user_id
and gr_cards.created_at_dayhour = gr_stats.created_at_dayhour
ORDER BY gr_cards.created_at_day, 
         gr_cards.created_at_dayhour,
         gr_cards.cum_amt_cards_per_dayhour ASC
'''

In [46]:
df = client_internal.dataframe_from_query(query)

In [48]:
df[df["cum_amt_cards_last_seven_h"]>2]

,user_id,created_at_day,created_at_dayhour,cum_amt_cards_per_dayhour,cum_amt_cards_last_seven_h,sum_usd_amount,card_type
4,29763f0b-ed9c-41d2-861e-caed7f51cbff,2020-05-07,2020-05-07 09:00:00+00:00,3,3,NaN,None
5,158bf422-8d71-40f3-928c-ba6a166bf7b9,2020-05-07,2020-05-07 10:00:00+00:00,1,3,6.19,visa
6,158bf422-8d71-40f3-928c-ba6a166bf7b9,2020-05-07,2020-05-07 10:00:00+00:00,2,3,6.19,visa
7,4dfb7bf4-53fc-4143-bf91-e15a1cb3b3dd,2020-05-07,2020-05-07 10:00:00+00:00,2,3,6.19,master_card
8,4dfb7bf4-53fc-4143-bf91-e15a1cb3b3dd,2020-05-07,2020-05-07 10:00:00+00:00,3,3,6.19,master_card
...,...,...,...,...,...,...,...
1812676,47aeb17b-eb76-4a63-972b-d9797ff198d2,2022-05-13,2022-05-13 07:00:00+00:00,12,3,NaN,None
1812678,47aeb17b-eb76-4a63-972b-d9797ff198d2,2022-05-13,2022-05-13 07:00:00+00:00,13,3,NaN,None
1812679,47aeb17b-eb76-4a63-972b-d9797ff198d2,2022-05-13,2022-05-13 07:00:00+00:00,14,3,NaN,None
1812685,433c013f-3ab1-40af-b8db-fca3e19fa712,2022-05-13,2022-05-13 07:00:00+00:00,92,3,NaN,None


In [49]:
df[df["user_id"]=='a106ed76-f18c-44b6-9923-77b01dd11a99']

,user_id,created_at_day,created_at_dayhour,cum_amt_cards_per_dayhour,cum_amt_cards_last_seven_h,sum_usd_amount,card_type
1707272,a106ed76-f18c-44b6-9923-77b01dd11a99,2022-04-08,2022-04-08 16:00:00+00:00,1,2,NaN,None
1707330,a106ed76-f18c-44b6-9923-77b01dd11a99,2022-04-08,2022-04-08 16:00:00+00:00,2,2,NaN,None
1708980,a106ed76-f18c-44b6-9923-77b01dd11a99,2022-04-09,2022-04-09 05:00:00+00:00,3,1,56.75,master_card
1709884,a106ed76-f18c-44b6-9923-77b01dd11a99,2022-04-09,2022-04-09 14:00:00+00:00,4,1,5.00,master_card
1710905,a106ed76-f18c-44b6-9923-77b01dd11a99,2022-04-09,2022-04-09 21:00:00+00:00,5,2,NaN,None
1712204,a106ed76-f18c-44b6-9923-77b01dd11a99,2022-04-10,2022-04-10 11:00:00+00:00,6,1,NaN,None
1712338,a106ed76-f18c-44b6-9923-77b01dd11a99,2022-04-10,2022-04-10 12:00:00+00:00,7,2,NaN,None
1712996,a106ed76-f18c-44b6-9923-77b01dd11a99,2022-04-10,2022-04-10 17:00:00+00:00,8,3,NaN,None
1715958,a106ed76-f18c-44b6-9923-77b01dd11a99,2022-04-11,2022-04-11 18:00:00+00:00,9,1,147.75,master_card
1716190,a106ed76-f18c-44b6-9923-77b01dd11a99,2022-04-11,2022-04-11 19:00:00+00:00,10,2,141.84,master_card
